## 第12章　梯度向量 —— 数学工具的首次融合

> 本章目标：将第 11 章的偏导数打包成梯度向量 ∇f = [∂f/∂x₁, ..., ∂f/∂xₙ]ᵀ。验证两个核心几何性质——梯度指向最陡上升、梯度垂直于等高线——并用数值实验证明"沿负梯度走一步，函数值一定下降"。最后落地到 AI 核心公式：**θ = θ − lr · ∇L**，整个深度学习的训练循环就是这行公式的重复。
> 前置知识：第 11 章（偏导数）、第 6 章（矩阵）、第 4 章（向量）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪 - NumPy + Matplotlib')



### 12.1　打包偏导数 —— 梯度向量

偏导数逐个告诉你"往东走一步高度怎么变""往北走一步高度怎么变"，但你需要一个**统一的方向指示**——把所有偏导数打包成一个向量，这就是梯度。

$$∇f = \begin{bmatrix} \frac{\partial f}{\partial x_1} & \frac{\partial f}{\partial x_2} & \cdots & \frac{\partial f}{\partial x_n} \end{bmatrix}^T$$

梯度是一个**向量**——它不仅有大小（坡有多陡），还有方向（往哪走上升最快）。这个方向正是第 11.3 节的核心发现：**梯度方向 = 最陡上升方向 = 垂直于等高线的方向。** 这个简单的几何事实，是整个深度学习训练体系的基石。

📐 **定义　梯度（Gradient）**：∇f(x₁, ..., xₙ) = [∂f/∂x₁, ..., ∂f/∂xₙ]ᵀ。是一个与输入同维度的向量。梯度 = 0 意味着站在极值点（山顶或谷底或鞍点）——所有方向的偏导数同时为零。

💻 **代码　梯度 = 所有偏导数的向量，用数值方法验证**




In [ ]:
import numpy as np

def numerical_gradient(f, x, h=1e-5):
    """对任意 f: R^n → R 求数值梯度——返回与 x 同 shape 的梯度向量"""
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

# 测试：f(x,y) = x² + 10y²（狭长峡谷——SGD 的经典测试函数）
def f(x):
    return x[0]**2 + 10 * x[1]**2

x0 = np.array([3.0, 2.0])
grad_f = numerical_gradient(f, x0)

print(f"f(x) = x1² + 10·x2²")
print(f"在 x = {x0} 处:")
print(f"  梯度 ∇f = {grad_f}")
print(f"  解析: ∂f/∂x1 = 2x1 = {2*x0[0]:.1f}, ∂f/∂x2 = 20x2 = {20*x0[1]:.1f}")
print(f"  梯度大小 ‖∇f‖ = {np.linalg.norm(grad_f):.2f}")
print(f"  → 梯度方向 = 最陡上升方向, 梯度大小 = 坡的陡峭程度")




> **关键洞察**：`numerical_gradient` 对所有变量逐一扰动求偏导——这就是反向传播在做的事情，只不过反向传播用链式法则（第 13-14 章）一次性算出所有偏导数，比这个 for 循环快几万倍。但数学本质完全一样：**对每个参数独立求偏导，打包成梯度向量。**

🔗 **AI 连接**：PyTorch 的 `.backward()` 就是自动调用链式法则算出损失对所有参数的梯度，存储在每个参数的 `.grad` 属性中——`W.grad` 的形状和 `W` 完全相同，`W.grad[i,j]` 就是 `∂L/∂W[i,j]`。

---

### 12.2　沿负梯度走一步 —— 亲眼见证函数值下降

理论的验证需要一个实验：随机选一个出发点，沿负梯度方向走一小步——函数值是上升还是下降？

负梯度 −∇f 指向**最陡下降方向**——这是优化的圣杯。证明方法不靠公式推导，而是靠你在代码里亲眼看到：**`x_new = x_old − lr * ∇f(x_old)` → f(x_new) < f(x_old)。**

这个简单的更新规则，重复几十万次，就是训练一个神经网络的全过程。学习率 `lr` 控制在"最陡下降方向"上迈多大——太小收敛慢，太大会跳过最小值甚至发散。第 24 章会展示 Adam 等优化器如何自适应地调整每一步的有效学习率。

💻 **代码　沿负梯度方向迈一步：函数值必定下降**




In [ ]:
import numpy as np

def f(x):
    return x[0]**2 + 10 * x[1]**2  # 狭长峡谷

def numerical_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

x = np.array([3.0, 2.0])
print(f"起点 x = {x}, f(x) = {f(x):.2f}")

# 测试不同学习率
for lr in [0.01, 0.05, 0.2]:
    grad = numerical_gradient(f, x)
    x_new = x - lr * grad               # 沿负梯度迈一步
    f_new = f(x_new)
    decreased = f_new < f(x)
    print(f"  lr={lr:.2f}: f({x_new.round(3)}) = {f_new:.4f}  "
          f"下降={decreased}  {'✓' if decreased else '✗ (lr太大,跳过头了!)'}")

print(f"\n规则: lr 合适 → 必定下降; lr 太大 → 可能跳过最小值甚至上升")
print(f"这就是为什么调学习率是深度学习最重要的超参数之一")




> **关键洞察**：`lr=0.01` 迈小步，稳步下降；`lr=0.2` 步子太大，沿着峡谷壁弹跳，可能越过谷底。这个实验完美展示了学习率的核心权衡——第 24 章各种优化器的设计动机，就是"如何在不同阶段自动选择最优步长"。

---

### 12.3　代码证明：梯度垂直于等高线

第 11 章你看到了"梯度箭头垂直于等高线"的静态图。现在用代码**定量验证**这个几何性质：在等高线上任取一点，该点的梯度方向与等高线的切线方向做点积——应该精确为 0（垂直）。

💻 **代码　数值验证梯度⊥等高线**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f_xy(x, y):
    return x**2 + 2 * y**2

# 网格和梯度
x = np.linspace(-3, 3, 50); y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y); Z = f_xy(X, Y)
dZ_dy, dZ_dx = np.gradient(Z, y, x)

# 取一个等高线上的点——(x=1, 解 f(1,y)=level 找 y)
level = f_xy(1.0, 1.0)
# 等高线的切线方向: 垂直于 ∇f，即 [−∂f/∂y, ∂f/∂x]
xp, yp = 1.5, 0.5  # 随便取一个点
gx = 2 * xp          # ∂f/∂x = 2x  (解析梯度)
gy = 4 * yp          # ∂f/∂y = 4y
tangent = np.array([-gy, gx])  # 切线方向: 旋转 90°

# 梯度与切线的点积应为 0
grad = np.array([gx, gy])
dot_product = np.dot(grad, tangent)
print(f"在点 ({xp}, {yp}):")
print(f"  梯度 ∇f = [{gx:.2f}, {gy:.2f}]")
print(f"  等高线切线方向 = [{tangent[0]:.2f}, {tangent[1]:.2f}]")
print(f"  ∇f · tangent = {dot_product:.6f} ← 应为 0 (垂直!) ✓")

# 可视化：在一个更大的图上标注这个点
fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=15, cmap='viridis', alpha=0.5)
skip = 3
ax.quiver(X[::skip, ::skip], Y[::skip, ::skip],
          -dZ_dx[::skip, ::skip], -dZ_dy[::skip, ::skip],
          color='red', alpha=0.5, scale=60, width=0.004)
ax.quiver(xp, yp, gx, gy, angles='xy', scale_units='xy', scale=2,
          color='blue', width=0.04, label='∇f (梯度)')
ax.quiver(xp, yp, tangent[0], tangent[1], angles='xy', scale_units='xy', scale=8,
          color='green', width=0.03, label='切线方向')
ax.plot(xp, yp, 'ko', markersize=8)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
ax.set_title('梯度(蓝) ⊥ 切线(绿)，点积=0')
ax.legend(); plt.show()




---

### 12.4　AI 核心：损失函数的梯度 = 权重的更新方向

现在落到全书最重要的公式：

$$θ_{new} = θ_{old} − lr · ∇L(θ_{old})$$

这个公式就是整个深度学习训练的原子操作。L 是损失函数（标量），θ 是全体参数（可能上亿维的向量），∇L 是损失对所有参数的梯度。每一步：算梯度 → 沿负梯度方向走一小步 → 重复。几十万步后，参数逼近最优值——模型就训练好了。

**一元损失函数 L(w) = (w−3)² 的最优解是 w=3，梯度是 2(w−3)。** 从 w=0 出发，每一步 `w -= lr * 2*(w−3)`，w 逐渐收敛到 3。

💻 **代码　梯度下降模拟：从 w=0 到 w≈3，30 步收敛**




In [ ]:
import numpy as np

# L(w) = (w-3)²——最优解 w=3, 梯度 dL/dw = 2(w-3)
L = lambda w: (w - 3)**2
dL = lambda w: 2 * (w - 3)

w = 0.0       # 出发点——离最优解 3 很远
lr = 0.1
history = [w]

for step in range(30):
    w = w - lr * dL(w)   # θ = θ - lr * ∇L —— 训练循环的原子操作
    history.append(w)
    if step % 10 == 0 or step == 29:
        print(f"step {step:3d}: w={w:.4f}, L(w)={L(w):.6f}, "
              f"梯度={dL(w):.4f}")

print(f"\n最终: w={w:.4f} ≈ 真实最优 3.0  ✓")
print(f"梯度下降用 30 步从随机起点逼近了最优解")

# 可视化收敛过程
import matplotlib.pyplot as plt
w_grid = np.linspace(-1, 5, 200)
plt.figure(figsize=(8, 4))
plt.plot(w_grid, L(w_grid), 'b-', lw=2, label='L(w)=(w-3)²')
plt.plot(history, [L(w) for w in history], 'ro-', markersize=3, label='梯度下降轨迹')
plt.axvline(x=3, color='gray', linestyle='--', label='最优解 w=3')
plt.xlabel('w'); plt.ylabel('L(w)')
plt.title('梯度下降：30步从 w=0 收敛到 w≈3')
plt.legend(); plt.grid(alpha=0.3); plt.show()




> **关键洞察**：你刚刚写出了深度学习训练循环的最简形式——30 行代码包含了一整个 AI 训练过程的核心逻辑。真实训练不过是把 `w` 从标量换成上亿维参数向量，把 `dL` 从手工求导换成自动微分，把 `for step in range(30)` 换成 `for epoch in range(N)`——但数学本质完全一样：**每一步沿负梯度方向走一小步。**

🔗 **AI 连接**：第 24 章将展示 Adam 等优化器如何改进这个基础循环——`w -= lr * dL(w)` → `adam.step()`。第 30 章用这个模式训练一个真实的文本续写模型。第 14 章揭示 PyTorch 的 `.backward()` 如何自动算出 `dL(w)`。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）梯度 ∇f 是一个向量。它的方向和大小分别代表什么？什么情况下梯度为零向量？
2. （概念）为什么"沿负梯度方向走一步，函数值下降"只在学习率足够小时成立？学习率太大时会发生什么？
3. （代码）对 f(x,y) = sin(x²+y²) 定义在 [−2,2]×[−2,2] 上，画出等高线和梯度场。从 (1, 1) 出发，沿负梯度方向走 20 步（lr=0.1），在图上画出轨迹——观察轨迹如何"螺旋"进入一个极小值点。
---


> 🔗 **章末钩子**：梯度告诉你"往哪走"。但神经网络几十层深，最后一层的梯度怎么传递到第一层？需要把导数一层层拆包再乘起来——这就是链式法则。
> 预览：下一章——**链式法则**，反向传播的理论基础。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
